# Odometry Exercise 2 — plotting translation data

Load a timestamped translation log, plot what the encoders and local odometry reported, and compare final odometry displacement with your independent physical-distance measurements.

This notebook is an introduction to plotting odometry data. Start with the supplied synthetic example so that you can see what each cell produces. The example is not evidence about your robot. When you are ready, change the settings in the **Use your own data** cell and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV exported from the Serial Monitor, set `USE_EXAMPLE_DATA = False`, and enter its filename. This is the main cell you need to edit.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "odometry_exercise02_samples.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

Run this cell when using the example. You do not need to understand or edit the generation code. Its columns match the translation log in Exercise 2.


In [ ]:
example_rows = []
example_conditions = [
    (1, 100, 96, 1.0),
    (2, 200, 191, -2.0),
    (3, -150, -144, 1.5),
    (4, 300, 287, -3.0),
]

for trial_num, command_mm, final_x_mm, sideways_mm in example_conditions:
    for sample_num, progress in enumerate(np.linspace(0, 1, 31)):
        example_rows.append({
            "test_name": "translation",
            "trial_num": trial_num,
            "commanded_distance_mm": command_mm,
            "sample_time_ms": 1000 * trial_num + 50 * sample_num,
            "left_encoder_count": round(progress * final_x_mm * 3.55),
            "right_encoder_count": round(progress * final_x_mm * 3.58),
            "odometry_x_mm": progress * final_x_mm,
            "odometry_y_mm": sideways_mm * np.sin(np.pi * progress),
            "odometry_theta_rad": 0.02 * np.sin(np.pi * progress),
        })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

When `USE_EXAMPLE_DATA` is false, `pd.read_csv(...)` reads your file. The final line displays its first five rows. Check that they look like the rows you saved before continuing.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Calculate elapsed time

Each trial may begin at a different StampC3 timestamp. Subtracting the first timestamp in each trial makes every plot begin at zero seconds.


In [ ]:
data = data.loc[data["test_name"] == "translation"].copy()
data = data.sort_values(["trial_num", "sample_time_ms"])
data["elapsed_s"] = (
    data["sample_time_ms"]
    - data.groupby("trial_num")["sample_time_ms"].transform("first")
) / 1000
data["trial_label"] = "Trial " + data["trial_num"].astype(str)

data[["trial_num", "sample_time_ms", "elapsed_s"]].head()


## 5. Plot both encoder counts

The left and right traces should be read as recorded wheel motion, not as an independent measurement of physical distance. Look at their directions and whether they remain similar during each translation.


In [ ]:
encoder_plot_data = data.melt(
    id_vars=["trial_num", "trial_label", "elapsed_s"],
    value_vars=["left_encoder_count", "right_encoder_count"],
    var_name="wheel",
    value_name="encoder_count",
)

grid = sns.relplot(
    data=encoder_plot_data,
    x="elapsed_s",
    y="encoder_count",
    hue="wheel",
    col="trial_label",
    col_wrap=2,
    kind="line",
    marker="o",
    estimator=None,
    height=3.2,
)
grid.set_axis_labels("Elapsed time (s)", "Encoder count")
grid.set_titles("{col_name}")
grid.figure.suptitle("Encoder counts during each translation", y=1.03)
plt.show()


## 6. Plot the reported x-y paths

This is the trajectory reported by the local odometry model. It is not an external measurement of the physical path, but it can help you see whether the model reported sideways motion.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(
    data=data,
    x="odometry_x_mm",
    y="odometry_y_mm",
    hue="trial_label",
    marker="o",
    estimator=None,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set(
    title="Paths reported by local odometry",
    xlabel="Odometry x (mm)",
    ylabel="Odometry y (mm)",
)
plt.show()


## 7. Extract the initial and final reported positions

The following cell creates one row per trial from the uploaded log. It calculates the reported changes in x, y and heading from the initial and final odometry values.


In [ ]:
trial_summary = (
    data.groupby("trial_num", as_index=False)
    .agg(
        commanded_distance_mm=("commanded_distance_mm", "first"),
        initial_x_mm=("odometry_x_mm", "first"),
        final_x_mm=("odometry_x_mm", "last"),
        initial_y_mm=("odometry_y_mm", "first"),
        final_y_mm=("odometry_y_mm", "last"),
        initial_heading_rad=("odometry_theta_rad", "first"),
        final_heading_rad=("odometry_theta_rad", "last"),
    )
)
trial_summary["reported_distance_mm"] = (
    trial_summary["final_x_mm"] - trial_summary["initial_x_mm"]
)
trial_summary["reported_sideways_change_mm"] = (
    trial_summary["final_y_mm"] - trial_summary["initial_y_mm"]
)
trial_summary["reported_heading_change_deg"] = np.degrees(
    trial_summary["final_heading_rad"]
    - trial_summary["initial_heading_rad"]
)

trial_summary


## 8. Enter your physical distance measurements

The timestamped robot log does not contain the distance you measured on the printed sheet. Enter those signed measurements below. Use positive values for forward motion and negative values for reverse motion. Add or remove rows so that the trial numbers match your log. The model radius is the value used when that log was recorded; label each row as `calibration` or `held-back` before comparing results.


In [ ]:
if USE_EXAMPLE_DATA:
    physical_measurements = pd.DataFrame({
        "trial_num": [1, 2, 3, 4],
        "measured_distance_mm": [102, 204, -153, 306],
        "direction": ["forward", "forward", "reverse", "forward"],
        "model_wheel_radius_mm": [16.0, 16.0, 16.0, 16.0],
        "model_wheel_separation_mm": [90.0, 90.0, 90.0, 90.0],
        "surface": ["example surface"] * 4,
        "data_use": ["calibration", "calibration", "calibration", "held-back"],
    })
else:
    physical_measurements = pd.DataFrame({
        "trial_num": [1, 2],
        "measured_distance_mm": [np.nan, np.nan],
        "direction": ["forward", "reverse"],
        "model_wheel_radius_mm": [np.nan, np.nan],
        "model_wheel_separation_mm": [np.nan, np.nan],
        "surface": ["", ""],
        "data_use": ["calibration", "held-back"],
    })

comparison = trial_summary.merge(physical_measurements, on="trial_num")
comparison


## 9. Calculate a radius estimate for each calibration trial

Exercise 2 gives the calculation `new radius = old radius × measured distance ÷ reported distance`. The cell applies it only to rows you labelled `calibration`, then shows the individual estimates together with their mean and median.


In [ ]:
calibration_rows = comparison.loc[
    comparison["data_use"] == "calibration"
].copy()
calibration_rows["estimated_wheel_radius_mm"] = (
    calibration_rows["model_wheel_radius_mm"]
    * calibration_rows["measured_distance_mm"]
    / calibration_rows["reported_distance_mm"]
)

radius_summary = calibration_rows["estimated_wheel_radius_mm"].agg(
    ["mean", "median"]
)
display(calibration_rows[[
    "trial_num", "measured_distance_mm", "reported_distance_mm",
    "estimated_wheel_radius_mm",
]])
radius_summary


## 10. Plot measured distance against reported distance

Points on the dashed line indicate agreement. A consistent offset from the line suggests that the wheel-radius value used by the local model may need adjustment. Keep the held-back row out of the radius estimate and use it later to evaluate the selected value.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
sns.scatterplot(
    data=comparison,
    x="measured_distance_mm",
    y="reported_distance_mm",
    hue="trial_num",
    palette="viridis",
    s=100,
    legend=False,
    ax=ax,
)
ax.axline((0, 0), slope=1, color="black", linestyle="--")
ax.set(
    title="Physical measurement and odometry report",
    xlabel="Measured physical displacement (mm)",
    ylabel="Reported odometry displacement (mm)",
)
plt.show()


## What to notice

- Do the encoder counts change in the expected direction?
- Does the local odometry report appreciable sideways movement?
- Does the reported distance tend to be larger or smaller than the physical measurement?

Save the notebook with your plots. Base conclusions about physical distance on your independent measurements, not on the commanded value.
